# Assignment 4 – Pets Expression Classification
## Model: MobileNet (Scratch + Transfer Learning)
**Alexandria University – Faculty of Engineering**  
**CCE: Computer Vision**

---
### Covers:
- **Part 2** – MobileNet built from scratch with data augmentation → accuracy, precision, recall, F1, confusion matrix  
- **Part 3** – Transfer Learning using MobileNetV2 with ImageNet weights


## 0 · Install & Import Dependencies

In [ ]:
# Install required packages (run once)
# !pip install tensorflow numpy matplotlib seaborn scikit-learn

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

print("TensorFlow version:", tf.__version__)

## 1 · Configuration

In [ ]:
DATASET_DIR    = "./dataset/master"   # root: dataset/master/{train,valid,test}/{class}/
IMG_SIZE       = (224, 224)   # MobileNet standard input
BATCH_SIZE     = 32
EPOCHS_SCRATCH = 50           # max epochs when training from scratch
EPOCHS_TL      = 30           # max epochs for transfer-learning fine-tune
LEARNING_RATE  = 1e-3
SEED           = 42
ALPHA          = 1.0          # MobileNet width multiplier (1.0 = full capacity)

tf.random.set_seed(SEED)
np.random.seed(SEED)


## 2 · Data Loading & Augmentation

In [ ]:
def build_datasets(dataset_dir: str, img_size: tuple, batch_size: int):
    """
    Load images from a pre-split directory structure and return
    (train_ds, val_ds, test_ds, class_names).

    Expected directory structure:
        dataset/master/
            train/   sad/  angry/  happy/  other/
            valid/   sad/  angry/  happy/  other/
            test/    sad/  angry/  happy/  other/
    """
    import os

    train_dir = os.path.join(dataset_dir, "train")
    valid_dir = os.path.join(dataset_dir, "valid")
    test_dir  = os.path.join(dataset_dir, "test")

    # ── Data Augmentation pipeline (applied ONLY to training images) ──
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomBrightness(0.2),
        layers.RandomContrast(0.2),
    ], name="data_augmentation")

    # Normalise pixel values to [0, 1]
    normalise = layers.Rescaling(1.0 / 255)

    def preprocess_train(image, label):
        image = augmentation(image, training=True)
        image = normalise(image)
        return image, label

    def preprocess_eval(image, label):
        image = normalise(image)
        return image, label

    AUTOTUNE = tf.data.AUTOTUNE

    raw_train = keras.utils.image_dataset_from_directory(
        train_dir,
        image_size=img_size,
        batch_size=None,
        shuffle=True,
        seed=SEED,
        label_mode="int",
    )
    class_names = raw_train.class_names

    raw_val = keras.utils.image_dataset_from_directory(
        valid_dir,
        image_size=img_size,
        batch_size=None,
        shuffle=False,
        label_mode="int",
        class_names=class_names,   # enforce same order as train
    )

    raw_test = keras.utils.image_dataset_from_directory(
        test_dir,
        image_size=img_size,
        batch_size=None,
        shuffle=False,
        label_mode="int",
        class_names=class_names,
    )

    train_size = sum(1 for _ in raw_train)
    val_size   = sum(1 for _ in raw_val)
    test_size  = sum(1 for _ in raw_test)
    print(f"Classes       : {class_names}")
    print(f"Train / Val / Test : {train_size} / {val_size} / {test_size}")

    train_ds = (raw_train
                .map(preprocess_train, num_parallel_calls=AUTOTUNE)
                .batch(batch_size).prefetch(AUTOTUNE))
    val_ds   = (raw_val
                .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
                .batch(batch_size).prefetch(AUTOTUNE))
    test_ds  = (raw_test
                .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
                .batch(batch_size).prefetch(AUTOTUNE))

    return train_ds, val_ds, test_ds, class_names


# ── Build datasets ──────────────────────────────────────────────────
train_ds, val_ds, test_ds, class_names = build_datasets(
    DATASET_DIR, IMG_SIZE, BATCH_SIZE
)
n_classes   = len(class_names)
input_shape = IMG_SIZE + (3,)


### Visualise augmented samples

In [ ]:
# Plot a batch of augmented training images
sample_batch = next(iter(train_ds))
images, labels = sample_batch[0][:16], sample_batch[1][:16]

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for img, lbl, ax in zip(images, labels, axes.flat):
    ax.imshow(img.numpy())
    ax.set_title(class_names[int(lbl)], fontsize=9)
    ax.axis("off")
plt.suptitle("Augmented Training Samples", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3 · MobileNet Architecture (from Scratch)

MobileNet (Howard et al., 2017) replaces standard 3×3 convolutions with  
**Depthwise-Separable Convolutions** — a depthwise spatial conv followed by  
a 1×1 pointwise conv — reducing computation by ~8–9× with minimal accuracy loss.

```
Input (224×224×3)
│
▼
Conv2D / s2  →  32 filters
│
▼
13 × Depthwise-Separable blocks  (strides alternate 1 and 2)
      64 → 128 → 128 → 256 → 256 → 512(×6) → 1024 → 1024
│
▼
Global Average Pooling  →  Dropout  →  Dense (softmax)
```


In [ ]:
def _depthwise_separable_block(x, filters: int, strides: int, alpha: float, name: str):
    """
    Core MobileNet building block:
        DepthwiseConv2D  →  BN  →  ReLU6      (spatial filtering per channel)
        Conv2D 1×1       →  BN  →  ReLU6      (channel mixing / projection)

    alpha: width multiplier that scales the number of output filters.
    ReLU6 = min(max(0, x), 6)  — clamps activations, improves fixed-point performance.
    """
    filters = max(1, int(filters * alpha))

    # ── Depthwise convolution (one filter per input channel) ──
    x = layers.DepthwiseConv2D(
        kernel_size=3, strides=strides, padding="same",
        use_bias=False, name=f"{name}_dw")(x)
    x = layers.BatchNormalization(name=f"{name}_dw_bn")(x)
    x = layers.ReLU(6.0, name=f"{name}_dw_relu")(x)

    # ── Pointwise (1×1) convolution (mix channels) ──
    x = layers.Conv2D(
        filters, kernel_size=1, strides=1, padding="same",
        use_bias=False, name=f"{name}_pw")(x)
    x = layers.BatchNormalization(name=f"{name}_pw_bn")(x)
    x = layers.ReLU(6.0, name=f"{name}_pw_relu")(x)

    return x


def build_mobilenet_scratch(input_shape: tuple, n_classes: int, alpha: float = 1.0) -> Model:
    """
    MobileNet V1 from scratch — Howard et al. (2017).
    alpha controls the width multiplier (default 1.0 = full capacity).
    """
    inputs = keras.Input(shape=input_shape, name="input")

    # ── Stem: standard 3×3 Conv / stride 2 ──
    x = layers.Conv2D(int(32 * alpha), 3, strides=2, padding="same",
                      use_bias=False, name="conv0")(inputs)
    x = layers.BatchNormalization(name="conv0_bn")(x)
    x = layers.ReLU(6.0, name="conv0_relu")(x)

    # ── 13 Depthwise-Separable blocks ──
    # Each tuple: (output_filters, stride)
    dw_config = [
        (64,   1),
        (128,  2), (128,  1),
        (256,  2), (256,  1),
        (512,  2),
        (512,  1), (512,  1), (512,  1), (512,  1), (512,  1),  # 5 ×
        (1024, 2), (1024, 1),
    ]
    for i, (filters, strides) in enumerate(dw_config):
        x = _depthwise_separable_block(x, filters, strides, alpha, name=f"dw{i+1}")

    # ── Classifier head ──
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(n_classes, activation="softmax", name="predictions")(x)

    return Model(inputs, outputs, name="MobileNet_Scratch")


model_scratch = build_mobilenet_scratch(input_shape, n_classes, alpha=ALPHA)
model_scratch.summary()

## 4 · Callbacks

In [ ]:
def get_callbacks(checkpoint_path: str):
    """
    EarlyStopping   – stops training when val_accuracy stops improving (patience=8).
    ReduceLROnPlateau – halves LR when val_loss plateaus (patience=4).
    ModelCheckpoint – saves the best weights to disk.
    """
    return [
        EarlyStopping(monitor="val_accuracy", patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=4, min_lr=1e-7, verbose=1),
        ModelCheckpoint(filepath=checkpoint_path, monitor="val_accuracy",
                        save_best_only=True, verbose=0),
    ]

## 5 · Evaluation Utilities

In [ ]:
def evaluate_model(model: Model, test_ds, class_names: list, title: str) -> dict:
    """
    Collect predictions on the test set and report:
      • Accuracy
      • Macro Precision, Recall, F1-score
      • Per-class classification report
      • Confusion matrix heatmap (saved to PNG + displayed inline)
    """
    print(f"\n{'='*60}")
    print(f"  Evaluation: {title}")
    print(f"{'='*60}")

    y_true, y_pred = [], []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"\n  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}  (macro-averaged)")
    print(f"  Recall    : {rec:.4f}  (macro-averaged)")
    print(f"  F1-Score  : {f1:.4f}  (macro-averaged)")
    print(f"\n  Per-class report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    # ── Confusion Matrix ──
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(max(8, len(class_names)),
                                    max(6, len(class_names) - 1)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title(f"Confusion Matrix – {title}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    safe = title.replace(" ", "_").replace("/", "-")
    plt.savefig(f"confusion_matrix_{safe}.png", dpi=150)
    plt.show()

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


def plot_history(history, title: str):
    """Plot training & validation accuracy / loss curves side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history["accuracy"],     label="Train Acc")
    axes[0].plot(history.history["val_accuracy"], label="Val Acc")
    axes[0].set_title(f"{title} – Accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history["loss"],     label="Train Loss")
    axes[1].plot(history.history["val_loss"], label="Val Loss")
    axes[1].set_title(f"{title} – Loss")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    safe = title.replace(" ", "_").replace("/", "-")
    plt.savefig(f"training_history_{safe}.png", dpi=150)
    plt.show()

## 6 · Part 2 — Train MobileNet from Scratch

In [ ]:
model_scratch.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_scratch = model_scratch.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_SCRATCH,
    callbacks=get_callbacks("mobilenet_scratch_best.keras"),
    verbose=1,
)

### 6.1 · Training Curves

In [ ]:
plot_history(history_scratch, "MobileNet_Scratch")

### 6.2 · Test Set Evaluation

In [ ]:
results = {}
results["MobileNet_Scratch"] = evaluate_model(
    model_scratch, test_ds, class_names, "MobileNet Scratch"
)

## 7 · Part 3 — Transfer Learning (ImageNet Weights)

We use **MobileNetV2** (the improved successor) pre-trained on ImageNet as a frozen  
feature extractor, then attach a custom classification head.

### Training strategy — two phases:
| Phase | Base model | Learning rate | Goal |
|-------|-----------|---------------|------|
| 1 | Frozen | 1e-3 | Train new head quickly |
| 2 | Top 30 layers unfrozen | 1e-5 | Fine-tune high-level features |


In [ ]:
def build_mobilenet_transfer(input_shape: tuple, n_classes: int):
    """
    MobileNetV2 pre-trained on ImageNet as feature extractor.
    Custom Dense head is trained in Phase 1.
    Top 30 layers are unfrozen for Phase 2 fine-tuning.
    """
    base = keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,        # remove ImageNet classifier
        weights="imagenet",
    )
    base.trainable = False        # freeze all base layers (Phase 1)

    inputs = keras.Input(shape=input_shape, name="input")
    # MobileNetV2 preprocess_input scales pixels from [0,255] → [-1, 1]
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)  # training=False keeps BN in inference mode

    # ── Custom classification head ──
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, activation="relu", name="fc1")(x)
    x = layers.BatchNormalization(name="fc1_bn")(x)
    x = layers.Dropout(0.4, name="dropout")(x)
    outputs = layers.Dense(n_classes, activation="softmax", name="predictions")(x)

    model = Model(inputs, outputs, name="MobileNet_Transfer")
    return model, base


model_tl, base_model = build_mobilenet_transfer(input_shape, n_classes)
model_tl.summary()

### 7.1 · Phase 1 — Train head (base frozen)

In [ ]:
model_tl.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("[TL Phase 1] Training classification head — base frozen…")
history_tl_p1 = model_tl.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=get_callbacks("mobilenet_tl_phase1_best.keras"),
    verbose=1,
)

### 7.2 · Phase 2 — Fine-tune top layers

In [ ]:
# Unfreeze the entire base, then re-freeze all but the last 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable = sum(1 for l in base_model.layers if l.trainable)
print(f"Unfrozen layers in MobileNetV2 base: {trainable}")

# Lower LR is critical to avoid destroying pre-trained weights
model_tl.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("\n[TL Phase 2] Fine-tuning top layers…")
history_tl_p2 = model_tl.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TL,
    callbacks=get_callbacks("mobilenet_tl_phase2_best.keras"),
    verbose=1,
)

### 7.3 · Training Curves (both phases combined)

In [ ]:
class MergedHistory:
    """Concatenate two Keras History objects for unified plotting."""
    def __init__(self, h1, h2):
        self.history = {k: h1.history[k] + h2.history[k] for k in h1.history}

merged = MergedHistory(history_tl_p1, history_tl_p2)
plot_history(merged, "MobileNet_TransferLearning")

### 7.4 · Test Set Evaluation

In [ ]:
results["MobileNet_Transfer"] = evaluate_model(
    model_tl, test_ds, class_names, "MobileNet Transfer Learning"
)

## 8 · Final Results Summary

In [ ]:
print("\n" + "="*68)
print("  FINAL RESULTS SUMMARY — MobileNet")
print("="*68)
print(f"{'Model':<28} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-"*68)
for name, m in results.items():
    print(f"{name:<28} {m['accuracy']:>10.4f} {m['precision']:>10.4f}"
          f" {m['recall']:>10.4f} {m['f1']:>10.4f}")
print("="*68)

best = max(results, key=lambda k: results[k]["f1"])
print(f"\n★  Best variant by macro F1: {best}  (F1 = {results[best]['f1']:.4f})")